In [25]:
import numpy as np
import pandas as pd
from MarkovDecision import compare_strategies
import plotly.express as px
import plotly.graph_objects as go

In [26]:
traps = {
    "no_trap_layout": [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
    "few_traps_layout": [0,0,3,0,0,0,3,0,0,0,0,3,0,0,0],
    "many_traps_layout": [0, 0, 1, 1, 3, 2, 1, 3, 2, 1, 1, 0, 1, 1, 0],
    "two_in_a_row_layout": [0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0],
    "evil_fast_lane_layout": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 2, 3, 1, 0],
    "back_to_3_layout": [0,0,0,0,0,2,0,0,0,1,0,0,2,0,0],
}
for name, trap in traps.items():
    print(f"Testing layout: {name} with no circle")
    # out = markovDecision(trap, False)
    compare_strategies(trap, False)
    print("\n\n")
    print(f"Testing layout: {name} with circle")
    # out = markovDecision(trap, True)
    compare_strategies(trap, True)

Testing layout: no_trap_layout with no circle



Testing layout: no_trap_layout with circle
Testing layout: few_traps_layout with no circle



Testing layout: few_traps_layout with circle
Testing layout: many_traps_layout with no circle



Testing layout: many_traps_layout with circle
Testing layout: two_in_a_row_layout with no circle



Testing layout: two_in_a_row_layout with circle
Testing layout: evil_fast_lane_layout with no circle



Testing layout: evil_fast_lane_layout with circle
Testing layout: back_to_3_layout with no circle



Testing layout: back_to_3_layout with circle


In [27]:
def print_game_board(layout,circle, policy):
    RED = '\033[91m'      # Traps
    BLUE = '\033[94m'     # Goal
    CYAN = '\033[36m'     # Policy
    RESET = '\033[0m'

    def get_tile_parts(tid):
        trap_type = layout[tid-1]
        action = str(policy[tid-1]) if tid!=15 else "" 
        action_fmt = ""
        
            
        action_fmt = f"{CYAN} {action:^2} {RESET}"
        
        if tid == 15:
            tile_fmt = f"{BLUE} [] {RESET}"
        elif trap_type != 0:
            action_fmt = f"{CYAN}  {action:^2} {RESET}"
            tile_fmt = f"{RED} [{trap_type}] {RESET}"
        else:
            tile_fmt = f" [] "
            
        return action_fmt, tile_fmt

    row1_actions = []
    row2_tiles = []
    
    for i in range(1, 11):
        a, t = get_tile_parts(i)
        row1_actions.append(a)
        row2_tiles.append(t)
    
    ga, gt = get_tile_parts(15)
    
    main_actions = "   ".join(row1_actions) + "     " + ga
    main_tiles = " → ".join(row2_tiles) + "  →  " + gt
    main_tiles += " → " if circle else ""

    branch_row = " " * 18 + "↘" + " " * 52 + "↗"

    lane_actions = []
    lane_tiles = []
    for i in range(11, 15):
        a, t = get_tile_parts(i)
        lane_actions.append(a)
        lane_tiles.append(t)
        
    fast_actions = " " * 20 + "   ".join(lane_actions)
    fast_tiles = " " * 20+ " → ".join(lane_tiles)

    print("GAME LAYOUT + POLICY")
    print("=" * 75)
    print(main_actions)
    print(main_tiles)
    print(branch_row)
    print(fast_actions)
    print(fast_tiles)
    print("=" * 75)
    print(f"KEY: {CYAN}Top Label{RESET} = Optimal Action  | {RED}ID{RESET} = Trap")



In [28]:
def print_graphic(data, layout_name):
    fig = go.Figure()

    categories = ['Optimal', 'dice_1', 'dice_2', 'dice_3', 'dice_4', 'Uniform_random']
    test_values = [data[f"{c}_test"][0] for c in categories]
    test_errors = [data[f"{c}_test"][1] for c in categories]
    theo_values = [data[f"{c}_theoretical"] for c in categories]

    fig.add_trace(go.Bar(
        x=categories,
        y=test_values,
        error_y=dict(type='data', array=test_errors, visible=True),
        marker_color=["#4989b4","#aaaaaa","#aaaaaa","#aaaaaa","#aaaaaa","#aaaaaa"] ,  # Pass the list of colors here
        name='Experimental',
        showlegend=True,
        text=[f"‎ <br>{test_values[i]:.2f}<br>±{test_errors[i]:.2f}" for i in range(len(test_values))],
        textposition='auto',
        textangle=0,
        insidetextanchor='end',
        textfont=dict(color='black', size=12)
    ))

    fig.add_trace(
        go.Scatter(
            x=categories,
            y=theo_values,
            mode='markers+text',
            marker=dict(
                symbol='line-ew',
                size=60,
                line=dict(width=2, color="#FF8204")
            ),
            text=[f"{v:.2f}" for v in theo_values], 
            textposition="top center",
            textfont=dict(color="#BB5E02", size=12),
            name='Theoretical',
            cliponaxis=False
        )
    )

    fig.update_layout(
        title="Number of Turns for "+layout_name+" by Strategy, <br>Expected Value vs Average turns in 1000 games",
        xaxis_title="Strategy",
        yaxis_title="Number of turns",
        plot_bgcolor='white',
        paper_bgcolor='white',
        bargap=0.3,
        legend_title_text='',
        yaxis=dict(
            showticklabels=False, 
            showgrid=False,       
            title_standoff=10     
        ),
        xaxis=dict(
            showgrid=False,     
            zeroline = False
        )
    )


    fig.show()

# No Trap Layout

## Circle

In [41]:
data = compare_strategies(traps["no_trap_layout"],True)
print_game_board(layout = traps["no_trap_layout"],circle = True,policy = data["Optimal_policy"])
print_graphic(data, "no_trap_layout (circle)")

GAME LAYOUT + POLICY
 4      3      3      4      3      4      3      3      2      1           
 []  →  []  →  []  →  []  →  []  →  []  →  []  →  []  →  []  →  []   →   []  → 
                  ↘                                                    ↗
                     3      3      2      1  
                     []  →  []  →  []  →  [] 
KEY: Top Label = Optimal Action  | ID = Trap


## Not Circle

In [30]:
data = compare_strategies(traps["no_trap_layout"],False)
print_game_board(layout = traps["no_trap_layout"],circle = False,policy = data["Optimal_policy"])
print_graphic(data, "no_trap_layout (not circle)")

GAME LAYOUT + POLICY
 4      3      3      4      3      4      3      3      3      3           
 []  →  []  →  []  →  []  →  []  →  []  →  []  →  []  →  []  →  []   →   [] 
                  ↘                                                    ↗
                     4      4      3      3  
                     []  →  []  →  []  →  [] 
KEY: Top Label = Optimal Action  | ID = Trap


# Few traps layout 

## Circle

In [31]:
data = compare_strategies(traps["few_traps_layout"],True)
print_game_board(layout = traps["few_traps_layout"],circle = True,policy = data["Optimal_policy"])
print_graphic(data, "few_traps_layout (circle)")

GAME LAYOUT + POLICY
 4      4       4      3      3      4       3      3      2      1           
 []  →  []  →  [3]  →  []  →  []  →  []  →  [3]  →  []  →  []  →  []   →   []  → 
                  ↘                                                    ↗
                     3       3      2      1  
                     []  →  [3]  →  []  →  [] 
KEY: Top Label = Optimal Action  | ID = Trap


## Not Circle

In [32]:
data = compare_strategies(traps["few_traps_layout"],False)
print_game_board(layout = traps["few_traps_layout"],circle = False,policy = data["Optimal_policy"])
print_graphic(data, "few_traps_layout (not circle)")

GAME LAYOUT + POLICY
 4      4       3      3      3      3       3      3      3      3           
 []  →  []  →  [3]  →  []  →  []  →  []  →  [3]  →  []  →  []  →  []   →   [] 
                  ↘                                                    ↗
                     4       4      3      3  
                     []  →  [3]  →  []  →  [] 
KEY: Top Label = Optimal Action  | ID = Trap


# Many Traps Layout

## Circle

In [33]:
data = compare_strategies(traps["many_traps_layout"],True)
print_game_board(layout = traps["many_traps_layout"],circle = True,policy = data["Optimal_policy"])
print_graphic(data, "many_traps_layout (circle)")

GAME LAYOUT + POLICY
 2      1       1       1       1       1       1       1       1       1           
 []  →  []  →  [1]  →  [1]  →  [3]  →  [2]  →  [1]  →  [3]  →  [2]  →  [1]   →   []  → 
                  ↘                                                    ↗
                      1      1       1       1  
                     [1]  →  []  →  [1]  →  [1] 
KEY: Top Label = Optimal Action  | ID = Trap


## Not Circle

In [34]:
data = compare_strategies(traps["many_traps_layout"],False)
print_game_board(layout = traps["many_traps_layout"],circle = False,policy = data["Optimal_policy"])
print_graphic(data, "many_traps_layout (not circle)")

GAME LAYOUT + POLICY
 2      1       1       1       1       1       1       1       1       1           
 []  →  []  →  [1]  →  [1]  →  [3]  →  [2]  →  [1]  →  [3]  →  [2]  →  [1]   →   [] 
                  ↘                                                    ↗
                      1      1       1       1  
                     [1]  →  []  →  [1]  →  [1] 
KEY: Top Label = Optimal Action  | ID = Trap


# Two traps in a row Layout

## Circle

In [35]:
data = compare_strategies(traps["two_in_a_row_layout"],True)
print_game_board(layout = traps["two_in_a_row_layout"],circle = True,policy = data["Optimal_policy"])
print_graphic(data, "two_in_a_row_layout (circle)")

GAME LAYOUT + POLICY
 4       4       4      3      3      3      3      3      2      1           
 []  →  [2]  →  [2]  →  []  →  []  →  []  →  []  →  []  →  []  →  []   →   []  → 
                  ↘                                                    ↗
                     2      3       2       1  
                     []  →  []  →  [2]  →  [2] 
KEY: Top Label = Optimal Action  | ID = Trap


## Not Circle

In [36]:
data = compare_strategies(traps["two_in_a_row_layout"],False)
print_game_board(layout = traps["two_in_a_row_layout"],circle = False,policy = data["Optimal_policy"])
print_graphic(data, "two_in_a_row_layout (not circle)")

GAME LAYOUT + POLICY
 4       4       3      3      3      3      3      3      3      3           
 []  →  [2]  →  [2]  →  []  →  []  →  []  →  []  →  []  →  []  →  []   →   [] 
                  ↘                                                    ↗
                     4      4       4       3  
                     []  →  []  →  [2]  →  [2] 
KEY: Top Label = Optimal Action  | ID = Trap


# Evil Fast Lane Layout

##  Circle

In [37]:
data = compare_strategies(traps["evil_fast_lane_layout"],True)
print_game_board(layout = traps["evil_fast_lane_layout"],circle = True,policy = data["Optimal_policy"])
print_graphic(data, "evil_fast_lane_layout (circle)")

GAME LAYOUT + POLICY
 4      4      4      3      3      4      3      3      2      1           
 []  →  []  →  []  →  []  →  []  →  []  →  []  →  []  →  []  →  []   →   []  → 
                  ↘                                                    ↗
                      4       3       1       1  
                     [3]  →  [2]  →  [3]  →  [1] 
KEY: Top Label = Optimal Action  | ID = Trap


## Not Circle

In [38]:
data = compare_strategies(traps["evil_fast_lane_layout"],False)
print_game_board(layout = traps["evil_fast_lane_layout"],circle = False,policy = data["Optimal_policy"])
print_graphic(data, "evil_fast_lane_layout (not circle)")

GAME LAYOUT + POLICY
 4      4      3      4      3      3      3      3      3      3           
 []  →  []  →  []  →  []  →  []  →  []  →  []  →  []  →  []  →  []   →   [] 
                  ↘                                                    ↗
                      4       4       4       1  
                     [3]  →  [2]  →  [3]  →  [1] 
KEY: Top Label = Optimal Action  | ID = Trap


# Back to 3 layout

## Circle

In [39]:
data = compare_strategies(traps["back_to_3_layout"],True)
print_game_board(layout = traps["back_to_3_layout"],circle = True,policy = data["Optimal_policy"])
print_graphic(data, "back_to_3_layout (circle)")

GAME LAYOUT + POLICY
 3      1      3      4      3       4      2      3      1       1           
 []  →  []  →  []  →  []  →  []  →  [2]  →  []  →  []  →  []  →  [1]   →   []  → 
                  ↘                                                    ↗
                     3      3       2      1  
                     []  →  []  →  [2]  →  [] 
KEY: Top Label = Optimal Action  | ID = Trap


## Not Circle

In [40]:
data = compare_strategies(traps["back_to_3_layout"],False)
print_game_board(layout = traps["back_to_3_layout"],circle = False,policy = data["Optimal_policy"])
print_graphic(data, "back_to_3_layout (not circle)")

GAME LAYOUT + POLICY
 3      3      3      4      3       4      2      4      4       1           
 []  →  []  →  []  →  []  →  []  →  [2]  →  []  →  []  →  []  →  [1]   →   [] 
                  ↘                                                    ↗
                     3      3       3      3  
                     []  →  []  →  [2]  →  [] 
KEY: Top Label = Optimal Action  | ID = Trap
